# 02 — Target Construction

This notebook walks through the time-series SUE target. It's the first hands-on
notebook in the project and demonstrates that the targets module works correctly
before any data pulling happens.

**What this notebook does:**
1. Builds a synthetic EPS panel and computes SUE on it, to confirm the math works.
2. Pulls a tiny real EPS sample (1–3 tickers from yfinance) and computes SUE on that.
3. Sanity-checks the SUE distribution: should be roughly bell-shaped with fat tails.
4. Demonstrates the leakage-detection helpers from `tests/test_no_leakage.py`.

**What this notebook deliberately does NOT do:**
- Pull the full universe (that's `notebooks/01_data_audit.ipynb`, to be written).
- Build any features (that's `notebooks/03_features_eda.ipynb`).
- Do any modeling.

**Stopping condition for this notebook:** SUE values for the synthetic panel match
expectations, the real-data sample produces sensible SUE values, and the leakage
tests pass when run via `pytest`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Make `src` importable from the notebook.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.targets import (
    fit_srwd,
    time_series_sue,
    time_series_sue_panel,
    proxy_analyst_sue,
    MIN_HISTORY_QUARTERS,
)

## Part 1 — Synthetic SUE

Build an EPS series with known properties (linear trend + seasonality + noise),
fit the SRWD model, and inspect what we get.

In [ ]:
rng = np.random.default_rng(42)
n_quarters = 32  # 8 years
dates = pd.date_range('2015-03-31', periods=n_quarters, freq='QE')

# Build EPS: base 1.0, growth 0.02 per quarter, mild Q4 seasonality, noise.
trend = 1.0 + 0.02 * np.arange(n_quarters)
quarter_idx = (dates.quarter - 1).to_numpy()
seasonal = 0.15 * np.cos(2 * np.pi * quarter_idx / 4.0)
noise = rng.normal(0, 0.08, size=n_quarters)
eps = pd.Series(trend + seasonal + noise, index=dates, name='eps')

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(eps.index, eps.values, marker='o')
ax.set_title('Synthetic quarterly EPS')
ax.set_ylabel('EPS')
ax.grid(alpha=0.3)
plt.tight_layout()

In [ ]:
fit = fit_srwd(eps)
print(f'SRWD fit: drift={fit.drift:.4f}, sigma={fit.sigma:.4f}, n_obs={fit.n_obs}, valid={fit.is_valid}')

# Expected drift, given growth of 0.02 per quarter, is YoY change = 4 * 0.02 = 0.08.
# We should see ~0.08 here, modulo noise.
assert abs(fit.drift - 0.08) < 0.02, 'Drift estimate should be near 0.08'
print('Drift estimate matches expectation (within noise tolerance).')

In [ ]:
# Compute SUE for a synthetic next-quarter actual that beats expectations by 1 sigma.
next_period_end = dates[-1] + pd.DateOffset(months=3)
expected_eps = eps.iloc[-4] + fit.drift
actual_beat_1sigma = expected_eps + fit.sigma

sue = time_series_sue(eps, actual_eps=actual_beat_1sigma, period_end=next_period_end)
print(f'Expected EPS (SRWD): {expected_eps:.4f}')
print(f'Actual EPS (1-sigma beat): {actual_beat_1sigma:.4f}')
print(f'Resulting SUE: {sue:.4f}')
assert abs(sue - 1.0) < 1e-6, 'SUE should be exactly 1.0 for a 1-sigma beat'
print('SUE = 1.0 as expected for a 1-sigma beat.')

## Part 2 — SUE on a synthetic panel of multiple firms

Now compute SUE for a panel of multiple firms across multiple quarters,
exercising `time_series_sue_panel` end-to-end.

In [ ]:
def make_synthetic_panel(n_firms=20, n_quarters=24, lag_days=35, seed=0):
    rng = np.random.default_rng(seed)
    rows = []
    for firm_i in range(n_firms):
        ticker = f'FIRM{firm_i:03d}'
        dates = pd.date_range('2015-03-31', periods=n_quarters, freq='QE')
        # Each firm gets a different trend, seasonality amplitude, and noise level.
        base = 0.5 + rng.uniform(0, 2.0)
        growth = rng.uniform(-0.005, 0.03)
        seasonal_amp = rng.uniform(0, 0.3)
        noise_sigma = rng.uniform(0.03, 0.15)
        quarter_idx = (dates.quarter - 1).to_numpy()
        eps_values = (
            base
            + growth * np.arange(n_quarters)
            + seasonal_amp * np.cos(2 * np.pi * quarter_idx / 4.0)
            + rng.normal(0, noise_sigma, size=n_quarters)
        )
        for period_end, eps_val in zip(dates, eps_values):
            rows.append({
                'ticker': ticker,
                'period_end': period_end,
                'eps': eps_val,
                'filed': period_end + pd.Timedelta(days=lag_days),
            })
    return pd.DataFrame(rows)

eps_panel = make_synthetic_panel()
ann_dates = eps_panel[['ticker', 'period_end', 'filed']].rename(columns={'filed': 'announcement_date'})
print(f'Panel shape: {eps_panel.shape}')
print(eps_panel.head())

In [ ]:
sue_panel = time_series_sue_panel(eps_panel, ann_dates)
print(f'SUE panel shape: {sue_panel.shape}')
print(f'SUE non-null count: {sue_panel["sue"].notna().sum()}')
print(f'SUE summary stats:')
print(sue_panel['sue'].describe())

In [ ]:
# Plot the SUE distribution. Should look roughly mean-zero and bell-shaped
# (because by construction our 'actuals' come from the same distribution as
# the SRWD fit). In real data, expect fat tails and a slight positive skew
# (firms guide consensus down, then beat).
fig, ax = plt.subplots(figsize=(8, 4))
sue_panel['sue'].dropna().hist(bins=40, ax=ax)
ax.set_title('SUE distribution on synthetic panel')
ax.set_xlabel('SUE')
ax.axvline(0, color='red', linestyle='--', alpha=0.7)
ax.grid(alpha=0.3)
plt.tight_layout()

print(f'SUE mean: {sue_panel["sue"].mean():.3f} (should be near 0)')
print(f'SUE std:  {sue_panel["sue"].std():.3f} (should be near 1 by construction)')

## Part 3 — Real-data smoke test

Pull EPS from yfinance for a handful of tickers. yfinance's `Ticker.earnings_dates`
and `Ticker.income_stmt` exposures are notoriously inconsistent — this cell may need
to be rewritten when the proper EDGAR puller is built (`src/edgar_pull.py`). For
now we use it just to confirm SUE produces sensible numbers on real EPS values.

**This is a temporary scaffold.** The proper data path is EDGAR companyfacts.

In [ ]:
# Lazy install — only needed for this section. Safe to skip the cell
# entirely if you just want to verify the synthetic path works.
try:
    import yfinance as yf
except ImportError:
    print('yfinance not installed. Run: pip install yfinance')
    yf = None

In [ ]:
# Three blue-chip tickers with long EPS history.
tickers_to_smoke_test = ['AAPL', 'MSFT', 'JNJ']

if yf is not None:
    real_rows = []
    for ticker in tickers_to_smoke_test:
        try:
            t = yf.Ticker(ticker)
            ed = t.get_earnings_dates(limit=40)
            if ed is None or len(ed) == 0:
                print(f'{ticker}: no earnings dates returned')
                continue
            ed = ed.dropna(subset=['Reported EPS']).copy()
            ed = ed.reset_index()
            # The 'Earnings Date' column from yfinance is the announcement
            # date; we treat it both as `filed` and `announcement_date` here
            # for this smoke test. The proper EDGAR path will give us
            # cleaner separation.
            for _, row in ed.iterrows():
                ann_date = pd.to_datetime(row['Earnings Date']).normalize()
                # period_end is approximately 30-50 days before announcement; we
                # don't have it directly from this endpoint, so we approximate as
                # ann_date - 35 days then snap to the nearest quarter end.
                approx_period_end = (ann_date - pd.Timedelta(days=35)).to_period('Q').end_time.normalize()
                real_rows.append({
                    'ticker': ticker,
                    'period_end': approx_period_end,
                    'eps': float(row['Reported EPS']),
                    'filed': ann_date,
                })
        except Exception as e:
            print(f'{ticker}: {e}')

    real_panel = pd.DataFrame(real_rows).drop_duplicates(['ticker', 'period_end'])
    real_panel = real_panel.sort_values(['ticker', 'period_end']).reset_index(drop=True)
    print(f'Real panel shape: {real_panel.shape}')
    print(real_panel.groupby('ticker').size())
    print(real_panel.head(10))
else:
    real_panel = None

In [ ]:
if real_panel is not None and len(real_panel) > 0:
    real_ann = real_panel[['ticker', 'period_end', 'filed']].rename(
        columns={'filed': 'announcement_date'}
    )
    real_sue = time_series_sue_panel(real_panel, real_ann)
    print(f'Real SUE non-null: {real_sue["sue"].notna().sum()} / {len(real_sue)}')
    print(real_sue.dropna(subset=["sue"]).tail(15))
else:
    print('Skipped real-data smoke test.')

## Part 4 — Verify leakage tests pass

Run the test suite. All 8 tests should pass. If any fail, **stop here** and fix
before moving on.

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '-m', 'pytest', 'tests/test_no_leakage.py', '-v'],
    capture_output=True, text=True, cwd=str(REPO_ROOT),
)
print(result.stdout)
if result.returncode != 0:
    print('TESTS FAILED')
    print(result.stderr)
    raise RuntimeError('Leakage tests failed — fix before continuing')
else:
    print('All leakage tests passed.')

## Next steps

1. **Build `src/universe.py`**: scrape historical S&P 500 membership with add/remove
   dates from Wikipedia or a similar source. Handoff: a function that returns the
   list of tickers that were members of the S&P 500 as of any given date.
2. **Build `src/edgar_pull.py`**: pull `companyfacts` JSON from
   `https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json` for each ticker.
   Extract quarterly EPS (`EarningsPerShareDiluted`, `EarningsPerShareBasic`),
   period end dates, and **filing dates**. Cache to `data/raw/`.
3. **Run this notebook end-to-end on the real panel**, replacing the yfinance
   smoke test with the EDGAR-sourced panel.
4. **Run `assert_panel_no_future_features` and `assert_announcement_date_after_period_end`**
   from `tests/test_no_leakage.py` against the real panel.
5. Then proceed to feature construction.

If you're handing off to Claude Code at this point, the README plus this notebook
should be enough orientation. Tell it to start by building `src/universe.py` and
`src/edgar_pull.py`, and to keep the leakage tests passing throughout.